# 03. Dense Neural Network Experiments
**Objective:** Implement, train, and evaluate a Dense Neural Network for binary sentiment classification using the GoEmotions dataset.# 03 Dense RNN LSTM - To be implemented

In [1]:
import sys
import os
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from torch.utils.data import DataLoader, TensorDataset

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.models import DenseNN
from src.train import train_model
from src.evaluate import evaluate_model, print_metrics
from src.visualize import plot_training_curves, plot_confusion_matrix

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [6]:
train_df = pd.read_csv('../data/processed/train.csv')
test_df = pd.read_csv('../data/processed/test.csv')
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
print(train_df['label'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/train.csv'

In [7]:
#convert text to fixed-size vectors using
vectorizer = CountVectorizer(max_features=5000, stop_words='english')

X_train_bow = vectorizer.fit_transform(train_df['text']).toarray()
X_test_bow = vectorizer.transform(test_df['text']).toarray()

y_train = train_df['label'].values
y_test = test_df['label'].values

input_dim = X_train_bow.shape[1]
print(f"Input dimension (Vocabulary size): {input_dim}")

NameError: name 'train_df' is not defined

In [ ]:
X_train_tensor = torch.FloatTensor(X_train_bow)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test_bow)
y_test_tensor = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
model_dense = DenseNN(
    input_dim=input_dim,
    hidden_dims=[256, 128],
    output_dim=1,
    dropout_rate=0.3
).to(device)

history_dense = train_model(
    model=model_dense,
    train_loader=train_loader,
    val_loader=test_loader, #using test as validation
    epochs=20,
    lr=0.001,
    patience=4,
    device=device
)

print("Training completed.")

In [ ]:
metrics_dense = evaluate_model(model_dense, test_loader, device=device)
print_metrics(metrics_dense, model_name="DenseNN (BoW)")

In [ ]:
#save training curves
plot_training_curves(
    train_losses=history_dense['train_losses'],
    val_losses=history_dense['val_losses'],
    model_name="DenseNN",
    save_path="../outputs/figures/dense_training_curves.png",
    show=True
)

#save confusion matrix
plot_confusion_matrix(
    confusion_matrix=np.array(metrics_dense['confusion_matrix']),
    class_names=["Negative", "Positive"],
    model_name="DenseNN",
    save_path="../outputs/figures/dense_confusion_matrix.png",
    show=True
)